# 🌸 GreenNest Bloom: Wedding Venue Inpainting LoRA Training
Train a custom diffusion inpainting LoRA adapter on your 66 extracted wedding venue architecture images.

**Hardware requirement**: Free Google Colab **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).
**Estimated training time**: ~15 minutes.

In [ ]:
# 1. Verify GPU and install required ML libraries
import torch
print('CUDA GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
else:
    print('⚠️ Please go to Runtime > Change runtime type > select T4 GPU!')

!pip install -q diffusers transformers accelerate peft safetensors torchvision

In [ ]:
# 2. Extract your venue dataset
# Note: Drag and drop wedding_venues_dataset.zip into the Colab Files tab on the left
import os, zipfile

if os.path.exists('wedding_venues_dataset.zip'):
    with zipfile.ZipFile('wedding_venues_dataset.zip', 'r') as z:
        z.extractall('.')
    print('✅ Dataset extracted successfully!')
elif os.path.exists('wedding_venues'):
    print('✅ Dataset folder already present!')
else:
    print('📁 Upload wedding_venues_dataset.zip using the Files tab on the left, then re-run this cell.')

img_dir = 'wedding_venues/processed_1024'
if os.path.exists(img_dir):
    imgs = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]
    print(f'Found {len(imgs)} normalized 1024x1024 training photos ready for training.')

In [ ]:
# 3. Download official Diffusers SDXL LoRA trainer
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora_sdxl.py -O train_dreambooth_lora_sdxl.py

# 4. Start LoRA Training (800 steps, ~15 mins on free T4)
!accelerate launch train_dreambooth_lora_sdxl.py \
  --pretrained_model_name_or_path='diffusers/stable-diffusion-xl-1.0-inpainting-0.1' \
  --instance_data_dir='wedding_venues/processed_1024' \
  --output_dir='./venue_lora_weights' \
  --instance_prompt='a photo of a luxury wedding venue stage, [wedding_stage_venue]' \
  --resolution=1024 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --learning_rate=1e-4 \
  --lr_scheduler='constant' \
  --lr_warmup_steps=0 \
  --max_train_steps=800 \
  --mixed_precision='fp16' \
  --checkpointing_steps=200

In [ ]:
# 5. Download the trained LoRA weights file
import os
from google.colab import files

weights_file = './venue_lora_weights/pytorch_lora_weights.safetensors'
if os.path.exists(weights_file):
    print('🎉 Training finished successfully! Downloading weights...')
    files.download(weights_file)
else:
    print('Weights file not found yet. Check the training output above.')